# Huggingface pipeline

In [105]:
import importlib.metadata
import subprocess
import sys

def install_dep(modules: list):
    for pkg in modules:
        base = pkg.split("[")[0]  # strip extras like [torch]
        try:
            importlib.metadata.distribution(base)
            print(f"{pkg} already installed")
        except importlib.metadata.PackageNotFoundError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

modules = ["huggingface_hub",
           "datasets", 
           "transformers", 
           "torch", 
           "numpy", 
           "nvidia-ml-py", 
           "transformers[torch]", 
           "evaluate",
           "ipywidgets",
           "scikit-learn",
           "scipy"]
install_dep(modules)

huggingface_hub already installed
datasets already installed
transformers already installed
torch already installed
numpy already installed
nvidia-ml-py already installed
transformers[torch] already installed
evaluate already installed
ipywidgets already installed
scikit-learn already installed
scipy already installed


Login with notebook to Huggingface

In [106]:
from huggingface_hub import notebook_login
notebook_login()

In [107]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [108]:
from transformers import AutoTokenizer, DataCollatorWithPadding
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_function(item):
    return tokenizer(item["sentence1"],item["sentence2"],truncation=True,padding=True,return_tensors="pt")
data_collactor = DataCollatorWithPadding(tokenizer=tokenizer)

In [109]:
train_dataset = raw_datasets["train"]
train_dataset = train_dataset.map(tokenizer_function,batched=True)
train_dataset = train_dataset.remove_columns(["sentence1","sentence2","idx"])
train_dataset = train_dataset.rename_column("label","labels")
train_dataset.set_format("torch")

validation_dataset = raw_datasets["validation"]
validation_dataset = validation_dataset.map(tokenizer_function,batched=True)
validation_dataset = validation_dataset.remove_columns(["sentence1","sentence2","idx"])
validation_dataset = validation_dataset.rename_column("label","labels")
validation_dataset.set_format("torch")


In [110]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [111]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

In [112]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collactor,
    processing_class=tokenizer,
)

In [113]:
trainer.train()

/home/user/git/rigorous-ai/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
predictions = trainer.predict(validation_dataset)
print(predictions.predictions.shape, predictions.label_ids.shape)

/home/user/git/rigorous-ai/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


(408, 2) (408,)


In [104]:
import evaluate
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.3161764705882353, 'f1': 0.0}